# Bifurcación de canales — Formulación I: verificación formal del código
## Trabajo 01 · Estructura N.º 8 · Fase II

**Curso:** Flujo en superficie libre — UNSCH, Unidad de Posgrado

El enunciado exige **al menos tres pruebas documentadas de distinta naturaleza**, cada una con valor obtenido, valor esperado y tolerancia. Este cuaderno las ejecuta sobre el código de la Fase I, que se reutiliza **sin cambios**.

| N.º | Naturaleza | Qué se verifica | Estado |
|---|---|---|---|
| 1 | Solución analítica | Caso cuadrático α = π/2 y solución explícita del modo de diseño | ✅ |
| 2 | Caso publicado | El resolvedor frente al ejemplo numérico del libro de Castro-Orgaz y Hager (2019, §2.5) | ✅ con una salvedad (ver más abajo) |
| 3 | Propiedades matemáticas | Jacobiano, orden de convergencia, unicidad de la raíz, monotonía, límite | ✅ |

**Salvedad sobre la prueba 2.** El libro de Milne-Thomson no trae ejemplo numérico de §10.8, y no encontré otra fuente publicada que resuelva este mismo modelo de flujo ideal. El caso publicado que sí existe verifica **el resolvedor de Newton–Raphson** (el mismo código que usa la bifurcación) sobre un sistema no lineal 2×2 del libro, pero **no verifica las ecuaciones de la bifurcación**. La **validación física** del modelo (¿representa bien un canal real?) se hace en la Discusión, con datos experimentales publicados y con el modelo 2D. Conviene confirmar con el docente que este es un caso publicado aceptable.

## 1. Librerías y versiones

In [1]:
import sys, json
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

print('Python     :', sys.version.split()[0])
print('NumPy      :', np.__version__)
print('pandas     :', pd.__version__)
print('matplotlib :', matplotlib.__version__)

Python     : 3.13.16
NumPy      : 2.5.3
pandas     : 3.0.5
matplotlib : 3.11.2


## 2. Código de la Fase I (sin cambios)

Funciones del sistema de análisis (2×2) y de diseño (3×3), la función escalar $f(\chi)$, el valor inicial por bisección y el Newton–Raphson amortiguado. **Única adición:** `valor_inicial_diseno` (arranque por bisección sobre $\mu$ para el modo de diseño), que se explica en la sección 3b.

In [2]:
def exponente(alpha):
    """e = 1 - pi/alpha. Para 0 < alpha < pi resulta negativo."""
    if not (0.0 < alpha < np.pi):
        raise ValueError("El modelo exige 0 < alpha < pi (radianes).")
    return 1.0 - np.pi/alpha


def F_analisis(x, lam, mu, alpha):
    """Modo A. x = [u1, u2]. Devuelve None si el estado no es admisible (u <= 0)."""
    u1, u2 = x
    if u1 <= 0 or u2 <= 0:
        return None
    e = exponente(alpha)
    return np.array([lam*u1 + mu*u2 - 1.0,
                     lam*u1**e - mu*u2**e - 1.0])


def J_analisis(x, lam, mu, alpha):
    u1, u2 = x
    e = exponente(alpha)
    return np.array([[lam,              mu],
                     [lam*e*u1**(e-1), -mu*e*u2**(e-1)]])


def F_diseno(x, lam, phi, alpha):
    """Modo B. x = [u1, u2, mu]."""
    u1, u2, mu_ = x
    if u1 <= 0 or u2 <= 0 or mu_ <= 0:
        return None
    e = exponente(alpha)
    return np.array([lam*u1 + mu_*u2 - 1.0,
                     lam*u1**e - mu_*u2**e - 1.0,
                     mu_*u2 - phi])


def J_diseno(x, lam, phi, alpha):
    u1, u2, mu_ = x
    e = exponente(alpha)
    return np.array([[lam,              mu_,               u2],
                     [lam*e*u1**(e-1), -mu_*e*u2**(e-1),  -u2**e],
                     [0.0,              mu_,               u2]])

In [3]:
def f_escalar(chi, lam, mu, alpha):
    e = exponente(alpha)
    return lam*(chi/lam)**e - mu*((1 - chi)/mu)**e - 1.0


def chi_biseccion(lam, mu, alpha, n_pasos=8):
    """Aproxima la raiz unica de f(chi) en (0,1) por biseccion."""
    lo, hi = 1e-12, 1 - 1e-12
    for _ in range(n_pasos):
        m = 0.5*(lo + hi)
        if f_escalar(m, lam, mu, alpha) > 0:
            lo = m
        else:
            hi = m
    return 0.5*(lo + hi)


def valor_inicial(lam, mu, alpha, n_pasos=8):
    chi0 = chi_biseccion(lam, mu, alpha, n_pasos)
    return np.array([chi0/lam, (1 - chi0)/mu])


def newton_amortiguado(F, J, x0, tol=1e-12, max_iter=60, paso_min=1e-6):
    """Newton-Raphson con retroceso. Devuelve (x, tabla de iteraciones, convergio)."""
    x = np.array(x0, dtype=float)
    if F(x) is None:
        raise ValueError("El valor inicial es fisicamente inadmisible.")
    filas, lam_paso = [], np.nan
    for k in range(max_iter + 1):
        Fx = F(x)
        normF = np.max(np.abs(Fx))
        filas.append({"Iteración": k, **{f"x{j+1}": x[j] for j in range(len(x))},
                      "‖F‖∞": normF, "λk (paso)": lam_paso})
        if normF < tol:
            return x, pd.DataFrame(filas), True
        dx = np.linalg.solve(J(x), -Fx)
        lam_paso = 1.0
        while lam_paso >= paso_min:
            x_nuevo = x + lam_paso*dx
            F_nuevo = F(x_nuevo)
            if F_nuevo is not None and np.max(np.abs(F_nuevo)) < (1 - 1e-4*lam_paso)*normF:
                break
            lam_paso /= 2
        else:
            return x, pd.DataFrame(filas), False
        x = x_nuevo
    return x, pd.DataFrame(filas), False

In [4]:
def jacobiano_numerico(F, x, h_rel=1e-6):
    """Jacobiano por diferencias centrales."""
    x = np.asarray(x, dtype=float)
    Jn = np.zeros((len(F(x)), len(x)))
    for j in range(len(x)):
        h = h_rel*max(1.0, abs(x[j]))
        e_ = np.zeros_like(x); e_[j] = h
        Jn[:, j] = (F(x + e_) - F(x - e_))/(2*h)
    return Jn


def mu_explicito(lam, phi, alpha):
    """Solucion cerrada del modo de diseno (None si la fraccion no es alcanzable)."""
    p = np.pi/alpha
    num = lam**p*(1 - phi)**(1 - p) - 1.0
    if num <= 0:
        return None
    return (num/phi**(1 - p))**(1/p)


def chi_cuadratica(lam, mu):
    """alpha = pi/2: raiz admisible de chi^2 - (1+lam^2+mu^2) chi + lam^2 = 0."""
    s = 1 + lam**2 + mu**2
    return 0.5*(s - np.sqrt(s**2 - 4*lam**2))


# --- FUNCION NUEVA respecto de la Fase I (ver la nota de la seccion 3b) ---
def h_mu(mu_, lam, phi, alpha):
    """Ecuacion (7) con chi = 1 - phi fijo, como funcion escalar de mu (estrictamente decreciente)."""
    e = exponente(alpha)
    return lam*((1 - phi)/lam)**e - mu_*(phi/mu_)**e - 1.0


def valor_inicial_diseno(lam, phi, alpha, n_pasos=8):
    """Arranque del modo de diseno: biseccion sobre mu (bracketing), luego u1 y u2 de la continuidad."""
    lo, hi = 1e-9, 1.0
    if h_mu(lo, lam, phi, alpha) <= 0:
        raise ValueError("Fraccion no alcanzable con ese lambda.")
    while h_mu(hi, lam, phi, alpha) > 0:
        hi *= 2.0
    for _ in range(n_pasos):
        m = 0.5*(lo + hi)
        if h_mu(m, lam, phi, alpha) > 0: lo = m
        else:                            hi = m
    mu0 = 0.5*(lo + hi)
    return np.array([(1 - phi)/lam, phi/mu0, mu0])

## 3. Prueba 1: contra solución analítica

### 1a. Caso cuadrático $\alpha=\pi/2$

Con $\alpha=\pi/2$ el exponente es $e=-1$ y la ecuación (7) se reduce a $\lambda^2/\chi-\mu^2/(1-\chi)=1$, es decir, $\chi^2-(1+\lambda^2+\mu^2)\chi+\lambda^2=0$. La raíz admisible es
$$\chi=\tfrac12\left[(1+\lambda^2+\mu^2)-\sqrt{(1+\lambda^2+\mu^2)^2-4\lambda^2}\right],$$
y la otra raíz es mayor que 1 (espuria). Los valores esperados salen de esta fórmula, escrita aparte del resolvedor. El resolvedor arranca de un **valor ingenuo** (velocidades iguales), no del arranque por bisección.

### 1b. Solución explícita del modo de diseño

Con $\mu u_2=\varphi$ y $\lambda u_1=1-\varphi$, la ecuación (7) da una solución cerrada para $\mu$ (Fase I, sección 13). Se compara con el Newton 3×3 sobre una **muestra aleatoria con semilla fija 42** de 300 configuraciones con $\alpha\in(25^\circ,150^\circ)$, $\lambda\in(0.5,1.5)$ y $\varphi\in(0.05,0.80)$. Solo cuentan las configuraciones alcanzables (existe $\mu>0$).

**Hallazgo y corrección.** En una primera ejecución con el arranque fijo $\mu_0=0.5$ de la Fase I, Newton **no convergió en 1 de 240 configuraciones** ($\alpha\approx37^\circ$, $\lambda\approx1.36$, $\varphi\approx0.77$). Funcionaba en el ejemplo de la Fase I, pero no es robusto. Se reemplazó por un arranque con bisección sobre $\mu$ (la ecuación (7) con $\chi=1-\varphi$ fijo es estrictamente decreciente en $\mu$, de modo que el intervalo siempre encierra la raíz), igual que en el modo de análisis.

In [5]:
# --- 1a. alpha = pi/2 ---
casos_cuad = [(1.0, 1.0), (0.7, 0.3), (1.0, 0.5), (0.5, 0.5), (1.0, 0.25)]
filas_1a = []
for (l_, m_) in casos_cuad:
    F_ = lambda x, l_=l_, m_=m_: F_analisis(x, l_, m_, np.pi/2)
    J_ = lambda x, l_=l_, m_=m_: J_analisis(x, l_, m_, np.pi/2)
    x_, t_, ok_ = newton_amortiguado(F_, J_, np.array([1/(l_ + m_)]*2))   # arranque ingenuo
    filas_1a.append({"λ": l_, "μ": m_, "χ obtenido": l_*x_[0], "χ esperado": chi_cuadratica(l_, m_),
                     "iter.": len(t_) - 1, "convergió": ok_})
t1a = pd.DataFrame(filas_1a)
t1a["|Δχ|"] = (t1a["χ obtenido"] - t1a["χ esperado"]).abs()
display(t1a.style.format({"λ": "{:.2f}", "μ": "{:.2f}", "χ obtenido": "{:.12f}",
                          "χ esperado": "{:.12f}", "|Δχ|": "{:.2e}"}).hide(axis="index"))
err_1a = t1a["|Δχ|"].max()
print(f"Caso λ = μ = 1: (3 − √5)/2 = {(3 - np.sqrt(5))/2:.12f}")

λ,μ,χ obtenido,χ esperado,iter.,convergió,|Δχ|
1.00,1.00,0.381966011250,0.381966011250,4,True,0.00e+00
0.70,0.30,0.423803331528,0.423803331528,5,True,1.67e-16
1.00,0.50,0.609611796798,0.609611796798,3,True,3.33e-16
0.50,0.50,0.190983005625,0.190983005625,6,True,2.78e-17
1.00,0.25,0.779304445366,0.779304445366,4,True,0.00e+00


Caso λ = μ = 1: (3 − √5)/2 = 0.381966011250


In [6]:
# --- 1b. Modo de diseno: Newton 3x3 vs solucion explicita (semilla fija 42) ---
rng = np.random.default_rng(42)
N = 300
alpha_s = np.radians(rng.uniform(25, 150, N))
lam_s   = rng.uniform(0.5, 1.5, N)
phi_s   = rng.uniform(0.05, 0.80, N)

n_alcanz, n_ok, dif = 0, 0, []
fallos = []
for a_, l_, p_ in zip(alpha_s, lam_s, phi_s):
    mu_ex = mu_explicito(l_, p_, a_)
    if mu_ex is None:
        continue                                   # fraccion no alcanzable con ese lambda
    n_alcanz += 1
    F_ = lambda x, l_=l_, p_=p_, a_=a_: F_diseno(x, l_, p_, a_)
    J_ = lambda x, l_=l_, p_=p_, a_=a_: J_diseno(x, l_, p_, a_)
    x0_ = valor_inicial_diseno(l_, p_, a_)         # arranque por biseccion sobre mu
    x_, t_, ok_ = newton_amortiguado(F_, J_, x0_)
    if ok_:
        n_ok += 1
        dif.append(abs(x_[2] - mu_ex)/mu_ex)
    else:
        fallos.append((np.degrees(a_), l_, p_))
err_1b = max(dif)
print(f"Configuraciones generadas: {N} | alcanzables: {n_alcanz} | Newton convergió: {n_ok}")
print(f"Máximo error relativo |μ_Newton − μ_explícito|/μ_explícito = {err_1b:.2e}")
if fallos:
    print("Casos donde Newton NO convergió (α°, λ, φ):")
    for f_ in fallos[:10]: print("   ", tuple(round(v, 3) for v in f_))

Configuraciones generadas: 300 | alcanzables: 240 | Newton convergió: 240
Máximo error relativo |μ_Newton − μ_explícito|/μ_explícito = 1.63e-12


## 4. Prueba 2: contra un caso publicado

**Fuente.** Castro-Orgaz, O. y Hager, W. H. (2019). *Shallow Water Hydraulics*. Springer, §2.5 «Numerical Solution of Energy–Momentum Equations for Gate Flow», pp. 132–134 (el libro de tu sílabo). El caso: compuerta con resalto rechazado, $h_1=1$ m y $h_3=0.5$ m. Las ecuaciones (2.227)–(2.228) son

$$
F_1=h_1+\frac{q^2}{2gh_1^2}-\left(h_2+\frac{q^2}{2gh_2^2}\right)=0,\qquad
F_2=\frac{h_2^2}{2}+\frac{q^2}{gh_2}-\left(\frac{h_3^2}{2}+\frac{q^2}{gh_3}\right)=0,
$$

con el jacobiano (2.236). **Resultado publicado:** $h_2=0.0778$ m y $q=0.3322$ m²/s, «en pleno acuerdo con la solución analítica» (2.213)–(2.214).

**Qué se verifica.** Se aplica **la misma función `newton_amortiguado` de la bifurcación** a este sistema y se compara con (i) los valores publicados, con tolerancia de **una unidad de la última cifra impresa** ($10^{-4}$), y (ii) la solución analítica del libro: de (2.213), $q^2/g=\tfrac12 h_2h_3(h_2+h_3)$, que sustituida en la energía (2.210) deja un polinomio cuya raíz espuria $h_2=h_1$ se descarta.

**Sobre la tolerancia.** Se fijó primero en la mitad de la última cifra ($5\times10^{-5}$, el criterio de redondeo). La primera ejecución mostró que el libro **trunca** en lugar de redondear: el valor exacto $h_2=0.077856$ m se imprime como 0.0778 (redondeado sería 0.0779). Por eso la tolerancia correcta frente a lo impreso es $10^{-4}$. La comprobación estricta es la que se hace contra la solución analítica del libro ($10^{-9}$).

**Qué no se verifica.** Las ecuaciones de la bifurcación. Esta prueba comprueba que el resolvedor, el criterio de parada y el amortiguamiento funcionan sobre un sistema no lineal real de canales.

In [7]:
g_ = 9.81
h1_g, h3_g = 1.0, 0.5

def F_compuerta(x):
    q, h2 = x
    if q <= 0 or h2 <= 0:
        return None
    F1 = h1_g + q**2/(2*g_*h1_g**2) - h2 - q**2/(2*g_*h2**2)
    F2 = h2**2/2 + q**2/(g_*h2) - h3_g**2/2 - q**2/(g_*h3_g)
    return np.array([F1, F2])

def J_compuerta(x):
    """Jacobiano (2.236) del libro, con Fr2^2 = q^2/(g h2^3)."""
    q, h2 = x
    Fr2_2 = q**2/(g_*h2**3)
    return np.array([[q/(g_*h1_g**2) - q/(g_*h2**2), -(1 - Fr2_2)],
                     [2*q/(g_*h2) - 2*q/(g_*h3_g),    h2*(1 - Fr2_2)]])

# Arranque en la rama supercritica (como en las diapositivas U3.04)
x_g, tabla_g, ok_g = newton_amortiguado(F_compuerta, J_compuerta, [0.30, 0.05])
q_g, h2_g = x_g

# Solucion analitica del libro: polinomio en h2 tras sustituir (2.213) en (2.210), sin la raiz h2 = h1
coef = [h3_g/h1_g**2, h3_g**2/h1_g**2 - 4, 4*h1_g - h3_g, -h3_g**2]   # cubico en h2
raices = np.roots(coef)
raices = [r.real for r in raices if abs(r.imag) < 1e-12 and r.real > 0 and abs(r.real - h1_g) > 1e-9]
h2_an = min(raices)
q_an = np.sqrt(0.5*g_*h2_an*h3_g*(h2_an + h3_g))

print(f"Convergió: {ok_g} ({len(tabla_g)-1} iteraciones)")
print(f"Newton    : h2 = {h2_g:.8f} m   q = {q_g:.8f} m²/s")
print(f"Analítica : h2 = {h2_an:.8f} m   q = {q_an:.8f} m²/s")
print("Publicado : h2 = 0.0778 m      q = 0.3322 m²/s   (Castro-Orgaz y Hager, 2019, p. 134)")
tabla_g.rename(columns={"x1": "q [m²/s]", "x2": "h2 [m]"}).style.format(
    {"q [m²/s]": "{:.8f}", "h2 [m]": "{:.8f}", "‖F‖∞": "{:.3e}", "λk (paso)": "{:.3g}"}).hide(axis="index")

Convergió: True (6 iteraciones)
Newton    : h2 = 0.07785561 m   q = 0.33216904 m²/s
Analítica : h2 = 0.07785561 m   q = 0.33216904 m²/s
Publicado : h2 = 0.0778 m      q = 0.3322 m²/s   (Castro-Orgaz y Hager, 2019, p. 134)


Iteración,q [m²/s],h2 [m],‖F‖∞,λk (paso)
0,0.30000000,0.05000000,8.803e-01,nan
1,0.30535047,0.06306122,2.533e-01,1
2,0.32496327,0.07407002,4.972e-02,1
3,0.33170558,0.07760908,3.069e-03,1
4,0.33216706,0.07785455,1.330e-05,1
5,0.33216904,0.07785561,2.498e-10,1
6,0.33216904,0.07785561,6.939e-18,1


**Nota sobre el coeficiente $k_1$.** Las ecuaciones (2.227)–(2.228) del libro no llevan coeficiente de corrección, es decir, equivalen a $k_1=1$. Con $k_1=0.99$ (valor usado en el cuaderno de la compuerta de bocatoma) el sistema da $q=0.3338$ m²/s, que **no coincide** con el 0.3322 publicado; con $k_1=1$ sí coincide. La celda siguiente lo muestra.

In [8]:
def resolver_compuerta(k1):
    def F(x):
        q, h2 = x
        if q <= 0 or h2 <= 0: return None
        return np.array([h1_g + k1*q**2/(2*g_*h1_g**2) - h2 - k1*q**2/(2*g_*h2**2),
                         h2**2/2 + k1*q**2/(g_*h2) - h3_g**2/2 - k1*q**2/(g_*h3_g)])
    def J(x):
        q, h2 = x
        return np.array([[k1*q/g_*(1/h1_g**2 - 1/h2**2), -1 + k1*q**2/(g_*h2**3)],
                         [2*k1*q/g_*(1/h2 - 1/h3_g),      h2 - k1*q**2/(g_*h2**2)]])
    return newton_amortiguado(F, J, [0.30, 0.05])[0]

filas_k = []
for k1 in (1.00, 0.99):
    q_k, h2_k = resolver_compuerta(k1)
    filas_k.append({"k1": k1, "q [m²/s]": q_k, "h2 [m]": h2_k,
                    "q publicado 0.3322 (±1e-4)": abs(q_k - 0.3322) < 1e-4,
                    "h2 publicado 0.0778 (±1e-4)": abs(h2_k - 0.0778) < 1e-4})
display(pd.DataFrame(filas_k).style.format({"k1": "{:.2f}", "q [m²/s]": "{:.6f}", "h2 [m]": "{:.6f}"}).hide(axis="index"))

k1,q [m²/s],h2 [m],q publicado 0.3322 (±1e-4),h2 publicado 0.0778 (±1e-4)
1.00,0.332169,0.077856,True,True
0.99,0.333842,0.077856,False,True


### Validación física con datos experimentales (pendiente)

Fuera de la verificación, el modelo ideal debe **validarse** con datos experimentales publicados de bifurcaciones a 90° (por ejemplo, la base de datos de Ibrahim et al., 2022, o Momplot et al., 2017). La celda siguiente queda lista: mientras no se completen los datos, aparece como **PENDIENTE** y no cuenta como prueba superada.

In [9]:
validacion = {
    "referencia": None,    # p. ej. "Ibrahim et al. (2022), J. Hydraul. Eng. 148, 04022015, tabla X"
    "lam_mu_alpha": None,  # (lambda, mu, alpha en radianes)
    "fraccion_ramal_medida": None,
    "incertidumbre": None,
}
if all(v is not None for v in validacion.values()):
    l_, m_, a_ = validacion["lam_mu_alpha"]
    F_v = lambda x: F_analisis(x, l_, m_, a_); J_v = lambda x: J_analisis(x, l_, m_, a_)
    x_v, _, _ = newton_amortiguado(F_v, J_v, valor_inicial(l_, m_, a_))
    print("Fracción al ramal: modelo =", round(1 - l_*x_v[0], 4), "| medida =", validacion["fraccion_ramal_medida"])
else:
    print("Validación experimental: PENDIENTE (faltan datos publicados con geometría y reparto medido).")

Validación experimental: PENDIENTE (faltan datos publicados con geometría y reparto medido).


## 5. Prueba 3: propiedades matemáticas que deben cumplirse siempre

| Propiedad | Qué comprueba |
|---|---|
| Balance | $\lambda u_1+\mu u_2=1$ en 300 soluciones aleatorias |
| Jacobiano | analítico igual a numérico en 100 puntos aleatorios, en los dos modos (error **relativo** $|J_{an}-J_{num}|/(1+|J_{an}|)$: con exponentes de hasta −8 las derivadas llegan a $10^{6}$ y el error absoluto de las diferencias finitas no es comparable) |
| Orden de convergencia | el orden observado de Newton es ≈ 2 (confirma que el jacobiano es exacto) |
| Unicidad | $f(\chi)$ decrece estrictamente y cambia de signo en 500 configuraciones aleatorias |
| Monotonía y límite | la fracción al ramal crece con $\mu$ y tiende a 0 cuando $\mu\to0$ |

Todas las muestras usan semilla fija 42.

In [10]:
rng = np.random.default_rng(42)

# 3a. Balance en 300 soluciones aleatorias (lambda, mu, alpha)
max_bal, n_noconv = 0.0, 0
for _ in range(300):
    l_, m_, a_ = rng.uniform(0.3, 1.5), rng.uniform(0.1, 1.5), np.radians(rng.uniform(20, 160))
    F_ = lambda x: F_analisis(x, l_, m_, a_); J_ = lambda x: J_analisis(x, l_, m_, a_)
    x_, _, ok_ = newton_amortiguado(F_, J_, valor_inicial(l_, m_, a_))
    n_noconv += (not ok_)
    max_bal = max(max_bal, abs(l_*x_[0] + m_*x_[1] - 1.0))

# 3b. Jacobiano analitico vs numerico en 100 puntos aleatorios, modos A y B
max_jac_A, max_jac_B = 0.0, 0.0
for _ in range(100):
    l_, m_, a_ = rng.uniform(0.3, 1.5), rng.uniform(0.1, 1.5), np.radians(rng.uniform(20, 160))
    xA = rng.uniform(0.2, 1.5, 2)
    xB = np.array([*rng.uniform(0.2, 1.5, 2), rng.uniform(0.1, 1.5)])
    FA_ = lambda x: F_analisis(x, l_, m_, a_)
    FB_ = lambda x: F_diseno(x, l_, 0.3, a_)
    JA_an, JB_an = J_analisis(xA, l_, m_, a_), J_diseno(xB, l_, 0.3, a_)
    max_jac_A = max(max_jac_A, (np.abs(JA_an - jacobiano_numerico(FA_, xA))/(1 + np.abs(JA_an))).max())
    max_jac_B = max(max_jac_B, (np.abs(JB_an - jacobiano_numerico(FB_, xB))/(1 + np.abs(JB_an))).max())

print(f"3a. Máx |λu1 + μu2 − 1| = {max_bal:.2e}  (no convergieron: {n_noconv} de 300)")
print(f"3b. Máx |J_an − J_num|/(1 + |J_an|): modo A = {max_jac_A:.2e}, modo B = {max_jac_B:.2e}")

3a. Máx |λu1 + μu2 − 1| = 2.22e-16  (no convergieron: 0 de 300)
3b. Máx |J_an − J_num|/(1 + |J_an|): modo A = 1.43e-09, modo B = 3.00e-10


In [11]:
# 3c. Orden de convergencia observado: p_k = ln(|F_{k+1}|/|F_k|) / ln(|F_k|/|F_{k-1}|)
ordenes = []
for (l_, m_, a_deg) in [(1.0, 0.5, 60), (1.0, 1.0, 90), (0.7, 0.3, 45), (1.0, 0.25, 120), (1.2, 0.8, 30)]:
    a_ = np.radians(a_deg)
    F_ = lambda x, l_=l_, m_=m_, a_=a_: F_analisis(x, l_, m_, a_)
    J_ = lambda x, l_=l_, m_=m_, a_=a_: J_analisis(x, l_, m_, a_)
    _, t_, _ = newton_amortiguado(F_, J_, np.array([1/(l_ + m_)]*2), tol=1e-14)
    nf = t_["‖F‖∞"].to_numpy()
    for k in range(1, len(nf) - 1):
        if nf[k+1] > 1e-14 and nf[k] < 1e-2:       # lejos del piso de redondeo y ya en el regimen asintotico
            ordenes.append(np.log(nf[k+1]/nf[k])/np.log(nf[k]/nf[k-1]))
ordenes = np.array(ordenes)
print(f"3c. Órdenes observados (n = {len(ordenes)}): mediana = {np.median(ordenes):.2f}, "
      f"mín = {ordenes.min():.2f}, máx = {ordenes.max():.2f}")

# 3d. Unicidad: f(chi) estrictamente decreciente y con cambio de signo, 500 configuraciones.
# Los extremos se evaluan con (chi, s = 1 - chi) por separado, porque 1 - chi pierde precision cuando chi ~ 1.
def f_cs(chi, s, lam, mu, alpha):
    e = exponente(alpha)
    return lam*(chi/lam)**e - mu*(s/mu)**e - 1.0

chis = np.linspace(1e-6, 1 - 1e-6, 2000)
potencias = 10.0**(-np.arange(1, 301))
n_mono, n_cambio, n_fuera_malla, extremos = 0, 0, 0, []
with np.errstate(over="ignore", invalid="ignore"):
    for _ in range(500):
        l_, m_, a_ = rng.uniform(0.2, 1.8), rng.uniform(0.05, 1.8), np.radians(rng.uniform(15, 165))
        fv = np.array([f_cs(c, 1 - c, l_, m_, a_) for c in chis])
        n_mono += bool(np.all(np.diff(fv) < 0))
        izq = f_cs(potencias, 1 - potencias, l_, m_, a_)        # chi -> 0
        der = f_cs(1 - potencias, potencias, l_, m_, a_)        # chi -> 1 (s dado exacto)
        n_cambio += bool(np.any(izq > 0) and np.any(der < 0))
        if not (fv[0] > 0 > fv[-1]):
            n_fuera_malla += 1
            extremos.append((np.degrees(a_), l_, m_))
print(f"3d. f decreciente en {n_mono}/500 configuraciones; cambio de signo (límites χ→0 y χ→1): {n_cambio}/500")
print(f"    En {n_fuera_malla}/500 la raíz queda a menos de 1e-6 de un extremo (fuera de la malla de 2000 puntos).")
for (a_deg, l_, m_) in extremos:
    print(f"      α = {a_deg:6.1f}°, λ = {l_:.2f}, μ = {m_:.2f}")

# 3e. Monotonia en mu y limite mu -> 0
n_mon_mu, n_tot = 0, 0
for _ in range(60):
    l_, a_ = rng.uniform(0.4, 1.6), np.radians(rng.uniform(20, 160))
    mus = np.linspace(0.02, 1.5, 40)
    frac = []
    for m_ in mus:
        F_ = lambda x: F_analisis(x, l_, m_, a_); J_ = lambda x: J_analisis(x, l_, m_, a_)
        x_, _, _ = newton_amortiguado(F_, J_, valor_inicial(l_, m_, a_))
        frac.append(m_*x_[1])
    n_tot += 1
    n_mon_mu += bool(np.all(np.diff(frac) > 0))
lim = []
for m_ in (1e-1, 1e-2, 1e-3, 1e-4):
    F_ = lambda x: F_analisis(x, 1.0, m_, np.pi/3); J_ = lambda x: J_analisis(x, 1.0, m_, np.pi/3)
    x_, _, _ = newton_amortiguado(F_, J_, valor_inicial(1.0, m_, np.pi/3))
    lim.append(m_*x_[1])
print(f"3e. Fracción al ramal creciente en μ: {n_mon_mu}/{n_tot} configuraciones")
print("    Límite μ→0 (λ=1, α=60°): fracción =", ", ".join(f"{v:.2e}" for v in lim), "para μ = 1e-1 … 1e-4")

3c. Órdenes observados (n = 5): mediana = 2.00, mín = 1.95, máx = 2.00
3d. f decreciente en 500/500 configuraciones; cambio de signo (límites χ→0 y χ→1): 500/500
    En 7/500 la raíz queda a menos de 1e-6 de un extremo (fuera de la malla de 2000 puntos).
      α =  157.1°, λ = 0.38, μ = 1.68
      α =  163.1°, λ = 0.28, μ = 1.38
      α =  163.7°, λ = 0.22, μ = 1.09
      α =  164.3°, λ = 0.36, μ = 1.20
      α =  162.1°, λ = 0.29, μ = 1.72
      α =  161.2°, λ = 1.78, μ = 0.20
      α =  164.9°, λ = 0.26, μ = 1.29


3e. Fracción al ramal creciente en μ: 60/60 configuraciones
    Límite μ→0 (λ=1, α=60°): fracción = 7.63e-02, 7.91e-03, 7.93e-04, 7.94e-05 para μ = 1e-1 … 1e-4


## 6. Tabla resumen de verificación

In [12]:
def fila(prueba, naturaleza, obtenido, esperado, tol, cumple):
    return {"Prueba": prueba, "Naturaleza": naturaleza, "Obtenido": obtenido,
            "Esperado": esperado, "Tolerancia": tol, "Estado": "CUMPLE" if cumple else "NO CUMPLE"}

err_pub_h2 = abs(h2_g - 0.0778); err_pub_q = abs(q_g - 0.3322)
err_an = max(abs(h2_g - h2_an), abs(q_g - q_an))
orden_med = float(np.median(ordenes))

filas = [
    fila("1a. χ cuadrático α = π/2 (5 casos, máx. |Δχ|)", "Analítica", err_1a, 0.0, 1e-10, err_1a < 1e-10),
    fila(f"1b. μ de diseño: Newton 3×3 vs explícito ({n_ok} casos, máx. error relativo)", "Analítica", err_1b, 0.0, 1e-9,
         err_1b < 1e-9 and n_ok == n_alcanz),
    fila("2a. Compuerta: |h2 − 0.0778| (libro, 2019)", "Publicada (resolvedor)", err_pub_h2, 0.0, 1e-4, err_pub_h2 < 1e-4),
    fila("2b. Compuerta: |q − 0.3322| (libro, 2019)", "Publicada (resolvedor)", err_pub_q, 0.0, 1e-4, err_pub_q < 1e-4),
    fila("2c. Compuerta: Newton vs analítica del libro", "Publicada (resolvedor)", err_an, 0.0, 1e-9, err_an < 1e-9),
    fila("3a. Balance λu1 + μu2 − 1 (300 casos)", "Propiedad matemática", max_bal, 0.0, 1e-12, max_bal < 1e-12 and n_noconv == 0),
    fila("3b. Jacobiano analítico vs numérico (error relativo), modo A", "Propiedad matemática", max_jac_A, 0.0, 1e-6, max_jac_A < 1e-6),
    fila("3b. Jacobiano analítico vs numérico (error relativo), modo B", "Propiedad matemática", max_jac_B, 0.0, 1e-6, max_jac_B < 1e-6),
    fila("3c. Orden de convergencia de Newton (mediana)", "Propiedad matemática", orden_med, 2.0, 0.3, abs(orden_med - 2) < 0.3),
    fila("3d. Unicidad: f decreciente (500 casos)", "Propiedad matemática", n_mono, 500, 0, n_mono == 500 and n_cambio == 500),
    fila("3e. Fracción al ramal creciente en μ (60 casos)", "Propiedad matemática", n_mon_mu, n_tot, 0, n_mon_mu == n_tot),
]
resumen = pd.DataFrame(filas)
resumen.loc[len(resumen)] = {"Prueba": "Validación experimental (no es verificación)", "Naturaleza": "Datos publicados",
                             "Obtenido": "—", "Esperado": "según la fuente", "Tolerancia": "por definir", "Estado": "PENDIENTE"}

def fmt(v):
    if isinstance(v, (float, np.floating)):
        return f"{v:.2e}" if (v != 0 and abs(v) < 1e-2) else f"{v:.4g}"
    return str(v)
vis = resumen.copy()
for c in ("Obtenido", "Esperado", "Tolerancia"):
    vis[c] = vis[c].map(fmt)
display(vis.style.hide(axis="index"))

n_ok_t = (resumen["Estado"] == "CUMPLE").sum(); n_fail = (resumen["Estado"] == "NO CUMPLE").sum()
display(Markdown(f"**{n_ok_t} de {len(resumen) - 1} comprobaciones de verificación cumplen** y {n_fail} no cumplen. "
                 f"La validación experimental sigue **pendiente** y es una etapa distinta."))

Prueba,Naturaleza,Obtenido,Esperado,Tolerancia,Estado
"1a. χ cuadrático α = π/2 (5 casos, máx. |Δχ|)",Analítica,3.33e-16,0,1.00e-10,CUMPLE
"1b. μ de diseño: Newton 3×3 vs explícito (240 casos, máx. error relativo)",Analítica,1.63e-12,0,1.00e-09,CUMPLE
"2a. Compuerta: |h2 − 0.0778| (libro, 2019)",Publicada (resolvedor),5.56e-05,0,1.00e-04,CUMPLE
"2b. Compuerta: |q − 0.3322| (libro, 2019)",Publicada (resolvedor),3.10e-05,0,1.00e-04,CUMPLE
2c. Compuerta: Newton vs analítica del libro,Publicada (resolvedor),5.55e-17,0,1.00e-09,CUMPLE
3a. Balance λu1 + μu2 − 1 (300 casos),Propiedad matemática,2.22e-16,0,1.00e-12,CUMPLE
"3b. Jacobiano analítico vs numérico (error relativo), modo A",Propiedad matemática,1.43e-09,0,1.00e-06,CUMPLE
"3b. Jacobiano analítico vs numérico (error relativo), modo B",Propiedad matemática,3.00e-10,0,1.00e-06,CUMPLE
3c. Orden de convergencia de Newton (mediana),Propiedad matemática,1.999,2,0.3,CUMPLE
3d. Unicidad: f decreciente (500 casos),Propiedad matemática,500,500,0,CUMPLE


**11 de 11 comprobaciones de verificación cumplen** y 0 no cumplen. La validación experimental sigue **pendiente** y es una etapa distinta.

## 7. Conclusiones de la Fase II y siguientes pasos

- El código de la bifurcación reproduce soluciones analíticas (caso cuadrático y solución explícita del diseño) con error del orden de la precisión de máquina.
- El **resolvedor** reproduce el ejemplo numérico publicado de la compuerta (Castro-Orgaz y Hager, 2019) y su solución analítica.
- Las propiedades matemáticas (jacobiano exacto, orden de convergencia ≈ 2, raíz única, monotonía y límite) se cumplen en muestras aleatorias con semilla fija.

**Limitaciones:**

- **La prueba con caso publicado verifica el resolvedor, no las ecuaciones de la bifurcación**, porque Milne-Thomson no trae ejemplo numérico. Hay que confirmar con el docente que esto es aceptable.
- Las comprobaciones analíticas verifican que el código resuelve bien la ecuación (7); **no** verifican que (7) represente bien un canal real. Eso es validación y se aborda en la Discusión con datos experimentales y con el modelo 2D.
- **Rango verificado del algoritmo:** $\alpha\in[20^\circ,160^\circ]$, $\lambda\in[0.3,1.5]$, $\mu\in[0.1,1.5]$ (prueba 3a, 300 casos, todos convergen). Fuera de ese rango no se garantiza nada: la prueba 3d encontró configuraciones con $\alpha$ cercano a $180^\circ$ cuya raíz queda a menos de $10^{-6}$ de un extremo de $(0,1)$; ahí la bisección inicial de la Fase I (límites $10^{-12}$) podría no encerrarla.
- **El arranque fijo $\mu_0=0.5$ de la Fase I para el modo de diseño no es robusto** (falló en 1 de 240 casos). La Fase II lo reemplaza por `valor_inicial_diseno`, que debe incorporarse al código final.
- La validación experimental está pendiente de obtener datos publicados con geometría y reparto medido.

**Fase III (siguiente):** análisis de sensibilidad de al menos cinco parámetros con elasticidades, sobre la Formulación II (con tirantes de control aguas abajo), que es la que aporta parámetros dimensionales como $n$, la pendiente o los tirantes de control.